#### Intrinsic Evaluations

Language modeling efficiency, which measures the mathematical divergence between the learned probability distribution and the true distribution of the training data.

#### 1. Entropy

Entropy measures how much information, on average, a token carries. Higher the entropy, the more information a token carries, and hence more bits are needed to represent a token. Hence we can say, entropy calculates how difficult it is to predict what comes next in a language. The lower a language’s entropy, i.e., the lesser the information a token carries, the more predictable that language.

Analogy: This is equivalent to saying, if you can perfectly predict what someone will say next, then what is said carries no new information. Entropy is highest when outcomes are equally likely and lowest when outcomes are deterministic. 

 - A highly structured domain (HTML, SQL, JSON) has lower entropy.
 - Casual conversation or creative texts have higher entropy



#### 2. Cross Entropy

The goal of training a LLM is to get the model to learn the distribution of the training data. A language model’s cross entropy on a dataset measures how difficult it is for the language model to predict what comes next in this data.

So if $P$ is the true distribution of the training data and $Q$ is the distribution learned by the language model, the cross-entropy  $H(P,Q)$  is expressed mathematically as:

$$H(P, Q) = H(P) + D_{KL}(P||Q)$$

$D_{KL}$ measure the diveregence, measuring how learned distribution diverges from true distribution


$$D_{KL}(P \| Q) = \sum_{x} P(x) \cdot \log \left(\frac{P(x)}{Q(x)}\right)$$

It measures the expected extra information required to encode samples drawn from the true distribution  P  when using an approximating distribution  Q . In simple terms, it quantifies how inefficient  Q  is at representing  P.

if P & Q were identical, then $\frac{P(x)} {Q(x)} = 1$ or 
$\log \left(\frac{P(x)} {Q(x)}\right) = 0$ 

hence $D_{KL}(P\|Q)$ = 0

A language model cannot achieve a cross-entropy lower than the inherent entropy of the dataset $( H(P) )$. This is the theoretical lower bound of performance.

Cross Entropy is not symmetrical, similarily KL Diveregence is not similar

---

#### 3. Perlexity 

It is exponeniated cross entropy measuring model's absolute uncertainity on predicitng next token. If measured in bits (base 2):

$$PPL(P, Q) = 2^{H(P, Q)}$$

Or logarithmic, amking perplexity exponential of e:
$$PPL(P, Q) = e^{H(P, Q)}$$

A lower perplexity value indicates that the model assigns a higher probability to the true token sequence, or we can also say, the more the uncertainty the model has in predicting what comes next in a given dataset, the higher the perplexity.

---


#### N-Gram Ovelap

#### 1. BLEU 

BLEU (Bilingual Evaluation Understudy) measures how many n-grams in the generated text appear in the reference (precision-focused). It is a metric primarily used to evaluate machine translation systems and produces a score between 0 and 1 (often reported as 0 to 100).

An 'n-gram' is just a way of describing 'n' consecutive words in a sentence. For instance, in the sentence "The cat is sleeping" we have bigrams (2-gram) as: "The cat", "cat is", "is sleeping". Also note that the words in an n-gram are taken in order, so jumbling up won't generate valid n-grams.


#### 2. ROUGE

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) does the converse for recall, commonly used in summarization. ROUGE is a family of metrics and unlike BLEU it measures how much of the reference text’s content is captured by the generated summary. Key variants of ROUGE:

 - ROUGE-N: Measures n-gram overlap (similar to BLEU but recall-based).
 - ROUGE-L: Uses the longest common subsequence (LCS) to capture sentence-level structure similarity.
 - ROUGE-S: Measures skip-bigram overlap (pairs of words in order but not necessarily adjacent).
 - Emphasizes coverage of important content from the reference summary.

Because summarization aims to include as much essential information as possible (rather than avoiding extra words), recall is often more important than precision in that context.

#### 3. Pairwise comparative evaluation and Elo ratings

Pairwise comparative evaluation is basically presenting two outputs for the same prompt to either a human or AI judge and ask which is better (or are they equal).

The outputs could be of different models, or different responses of the same model.

The Elo rating system is a mathematical method originally developed to rank chess players based on their performance against one another. Instead of simply counting wins and losses, Elo measures relative skill. When two players compete, the system predicts the probability of each player winning based on the difference in their ratings.

Elo works similarly, but instead of chess matches, models are compared through evaluation tasks. For example, two LLMs might be given the same prompt, and human evaluators (or automated systems) decide which response is better. Each comparison acts like a “match.”

If a lower-rated model produces a response that evaluators prefer over a higher-rated model’s response, it gains more rating points. Over thousands of pairwise comparisons, the ratings converge to reflect relative performance.

However, Elo has limitations when used for LLMs. It assumes that performance is transitive, i.e., if Model A beats B and B beats C, then A should beat C, which may not always hold across different task types. It also depends heavily on the quality and diversity of prompts and evaluators.

---

In [ ]:
source_text = """
The Amazon rainforest, often referred to as the "lungs of the Earth," produces
approximately 20 percent of the world's oxygen. Spanning over 5.5 million
square kilometers across nine countries in South America, it is the largest
tropical rainforest on the planet. The Amazon is home to an estimated 10
percent of all species on Earth, including over 40,000 plant species, 1,300
bird species, and 3,000 types of fish. Deforestation, primarily driven by
cattle ranching and soybean farming, poses the greatest threat to this
ecosystem. Between 2001 and 2020, the Amazon lost approximately 10 percent
of its forest cover. Scientists warn that continued deforestation could push
the rainforest past a tipping point, transforming large portions into savanna
and releasing billions of tons of stored carbon dioxide into the atmosphere.
"""

reference_summary = (
    "The Amazon rainforest is the world's largest tropical forest, producing "
    "about 20% of global oxygen and hosting 10% of Earth's species. "
    "Deforestation from agriculture threatens to push it past a tipping point, "
    "potentially converting it to savanna and releasing massive carbon emissions."
)

# Candidate A: a good summary that captures key points in different words
candidate_a = (
    "Spanning 5.5 million square kilometers, the Amazon is Earth's biggest "
    "tropical rainforest and a critical oxygen source. It harbors extraordinary "
    "biodiversity, but agricultural deforestation — mainly cattle and soy — "
    "risks triggering an irreversible shift to savanna, which would unleash "
    "vast carbon stores."
)

# Candidate B: a flawed summary — vague, misses key facts, adds a hallucination
candidate_b = (
    "The Amazon is a large forest in South America. It has many trees and "
    "animals. Some people cut down trees there. The forest was designated a "
    "UNESCO World Heritage Site in 2005."  # <-- hallucinated fact
)

#### BLEU Score

BLEU measures how many n-grams in the generated text appear in the reference (precision-focused).



In [2]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


def compute_bleu(candidate, reference):
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()

    # smoothing function
    smoothie = SmoothingFunction().method1
    return sentence_bleu([ref_tokens], cand_tokens, smoothing_function=smoothie)

bleu_a = compute_bleu(candidate_a, reference_summary)
bleu_b = compute_bleu(candidate_b, reference_summary)

print(f"BLEU - Candidate A: {bleu_a:.4f}")
print(f"BLEU - Candidate B: {bleu_b:.4f}")

BLEU - Candidate A: 0.0147
BLEU - Candidate B: 0.0123


#### ROUGE Score

ROUGE measures recall: how much of the reference content is captured by the candidate? This is especially relevant for summarization.



In [3]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_rouge(candidate, reference):
    scores = scorer.score(reference, candidate)
    return {key: round(val.fmeasure, 4) for key, val in scores.items()}

rouge_a = compute_rouge(candidate_a, reference_summary)
rouge_b = compute_rouge(candidate_b, reference_summary)

print(f"ROUGE — Candidate A: {rouge_a}")
print(f"ROUGE — Candidate B: {rouge_b}")

ROUGE — Candidate A: {'rouge1': 0.3908, 'rouge2': 0.0706, 'rougeL': 0.2529}
ROUGE — Candidate B: {'rouge1': 0.2368, 'rouge2': 0.027, 'rougeL': 0.1579}


#### BERT Score

To measure the semantic similarity between reference and candidates, which might have been missed by n-gram. it helps in recognizing the paraphrasing & wordings similarity

 - Precision: for every candidate token, we find the maximum cosine similarity with all reference tokens, then average these values across candidate tokens.
 
 - Recall is computed symmetrically: for every reference token, we find the maximum similarity with all candidate tokens, then average across reference tokens.

 - F1 Score is the harmonic mean of precision and recall is calculated to provide a balanced score.

In [4]:
from bert_score import score as bert_score

def compute_bertscore(candidates, references):
    P, R, F1 = bert_score(candidates, references, model_type="roberta-large", lang="en", verbose=False)
    return P.tolist(), R.tolist(), F1.tolist()

# Compute scores
precision_scores, recall_scores, f1_scores = compute_bertscore(
    [candidate_a, candidate_b],
    [reference_summary, reference_summary]
)

# Candidate A
print(f"Candidate A — Precision: {precision_scores[0]:.4f}")
print(f"Candidate A — Recall:    {recall_scores[0]:.4f}")
print(f"Candidate A — F1:        {f1_scores[0]:.4f}")

print()

# Candidate B
print(f"Candidate B — Precision: {precision_scores[1]:.4f}")
print(f"Candidate B — Recall:    {recall_scores[1]:.4f}")
print(f"Candidate B — F1:        {f1_scores[1]:.4f}")

/Users/monusingh/work-share/code-blogs-articles/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 860.99it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loadi

Candidate A — Precision: 0.8948
Candidate A — Recall:    0.9078
Candidate A — F1:        0.9012

Candidate B — Precision: 0.8875
Candidate B — Recall:    0.8474
Candidate B — F1:        0.8670


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()  # Loads from .env in the project root
api_key = os.environ.get("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

os.environ["OPENROUTER_API_KEY"] = api_key

In [6]:
import litellm
import json

client = litellm.LiteLLM()

JUDGE_PROMPT = """You are an expert evaluator assessing the quality of text summaries.

Given a source text, a reference summary (gold standard), and a candidate summary evaluate
the candidate on three criteria.

For each criterion, assign a score of 0 (fail) to 1 (pass) and provide a
brief justification.

Compare against both the source text AND the reference summary.

**Criteria:**
1. **Accuracy** — Are all claims in the candidate summary factually supported by the source?
   Any hallucinated or fabricated information => accuracy=0.
2. **Completeness** — Does the candidate summary capture ALL the key points of the source given the reference?
   Any missing key points => completeness=0.
3. **Conciseness** — Is the candidate summary brief? Or is it too verbose compared to reference summary?
   Significantly extra verbosity in candidate, given the reference => conciseness=0.

### Response format
Respond with ONLY a valid JSON object (no markdown fences, no commentary):
{
  "accuracy":     {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "completeness": {"score": <0 or 1>, "justification": "<1-2 sentences>"},
  "conciseness":  {"score": <0 or 1>, "justification": "<1-2 sentences>"}
}
"""

DATA_PROMPT = """\
SOURCE TEXT:
{source}

REFERENCE SUMMARY:
{reference}

CANDIDATE SUMMARY:
{candidate}
"""

def llm_judge(source, reference, candidate, max_retries=2):

    for attempt in range(max_retries + 1):
        resp = client.chat.completions.create(
            model="openrouter/openai/gpt-4o-2024-08-06",
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": DATA_PROMPT.format(
                    source=source, reference=reference, candidate=candidate
                )},
            ],
            temperature=0.00,
            max_tokens=1000
        )

        text = resp.choices[0].message.content.strip()

        # Robust JSON extraction: handle markdown fences (if suppose the model adds them)
        if text.startswith("```"):
            text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()

        # Hard-validate JSON; retry once if invalid
        try:
            data = json.loads(text)
            # minimal schema check
            for k in ("accuracy", "completeness", "conciseness"):
                assert k in data and "score" in data[k] and "justification" in data[k]
                assert data[k]["score"] in (0, 1)
            return data
        except Exception:
            if attempt == max_retries:
                # return raw for debugging if it keeps failing
                return {"error": "invalid_json", "raw": text}

# overall score
def overall_score(data):
    scores = [data[k]["score"] for k in ("accuracy", "completeness", "conciseness")]
    average_score = sum(scores) / len(scores)
    print(f"\nAverage score: {average_score}")
    if data["accuracy"]["score"] == 1 and average_score >= 0.65:
        return "PASS"
    else:
        return "FAIL"


print("=" * 60)
print("JUDGE EVALUATION — CANDIDATE A")
print("=" * 60)
data1 = llm_judge(source_text, reference_summary, candidate_a)
print(json.dumps(data1, indent=2))
print ("\n Overall:")
print(overall_score(data1))


print("\n" + "=" * 60)
print("JUDGE EVALUATION — CANDIDATE B")
print("=" * 60)
data2 = llm_judge(source_text, reference_summary, candidate_b)
print(json.dumps(data2, indent=2))
print ("\n Overall:")
print(overall_score(data2))

JUDGE EVALUATION — CANDIDATE A
{
  "accuracy": {
    "score": 1,
    "justification": "The candidate summary accurately reflects the source text, including the size of the Amazon, its role in oxygen production, biodiversity, and the threat of deforestation."
  },
  "completeness": {
    "score": 1,
    "justification": "The candidate summary captures all key points from the source text, including the size, biodiversity, deforestation causes, and potential consequences."
  },
  "conciseness": {
    "score": 1,
    "justification": "The candidate summary is concise and comparable in length to the reference summary, without unnecessary verbosity."
  }
}

 Overall:

Average score: 1.0
PASS

JUDGE EVALUATION — CANDIDATE B
{
  "accuracy": {
    "score": 0,
    "justification": "The candidate summary inaccurately claims the Amazon was designated a UNESCO World Heritage Site in 2005, which is not supported by the source text."
  },
  "completeness": {
    "score": 0,
    "justification": "The 